# Documentation & Gouvernance
## Unity Catalog · Tags · Règles de Qualité

**Objectif :** Documenter et gouverner les données du catalogue sport_metrics.

**Les 4 niveaux implémentés :**

| Niveau | Outil | Description |
|---|---|---|
| 1 | COMMENT ON TABLE | Description métier des tables |
| 2 | COMMENT ON COLUMN | Description des colonnes clés |
| 3 | SET TAGS | Catégorisation : domaine, audience, PII, ML |
| 4 | ADD CONSTRAINT | Règles de qualité : NOT NULL, RANGE, POSITIVE |

**Conformité RGPD :**
Les colonnes PII (données personnelles) sont identifiées et taguées.
`dim_players_info` contient des données sensibles → `pii = true`

### Niveau 1 & 2: Descriptions tables et colonnes

In [0]:
-- ============================================
-- Descriptions des tables RAW
-- ============================================
COMMENT ON TABLE sport_metrics.raw.team_players_stats
IS 'Statistiques brutes des joueurs par match. Source : CSV. Chargement quotidien.';

COMMENT ON TABLE sport_metrics.raw.team_training_sessions
IS 'Sessions d entraînement brutes. Métriques physiologiques : FC, charge, fatigue. Source : CSV.';

COMMENT ON TABLE sport_metrics.raw.team_players_personal_info
IS 'Informations personnelles brutes des joueurs : biométrie, position, expérience. Source : CSV.';

COMMENT ON TABLE sport_metrics.raw.team_games_dataset
IS 'Statistiques collectives brutes par match. Résultats, tirs, rebonds. Source : CSV.';

COMMENT ON TABLE sport_metrics.raw.team_boxscores
IS 'Box scores bruts. Utilisé uniquement pour PLUS_MINUS dans staging.team_games_dataset.';

-- ============================================
-- Descriptions des tables MART
-- ============================================
COMMENT ON TABLE sport_metrics.mart.int_fatigue_index
IS 'Fatigue Index calculé par joueur et par session. 
Formule : 30% charge interne + 40% charge externe + 30% récupération.
Réutilisé par mart_physical_condition et mart_game.';

COMMENT ON TABLE sport_metrics.mart.mart_player
IS 'Performance individuelle par match. 
Croise stats techniques + biométrie + dernier entraînement.
Consommé par Power BI.';

COMMENT ON TABLE sport_metrics.mart.mart_game
IS 'Performance collective par match.
Inclut Fi_team : fatigue moyenne équipe avant match.
Consommé par Power BI.';

COMMENT ON TABLE sport_metrics.mart.mart_physical_condition
IS 'Table centrale SportMetrics. 
Croise entraînement + Fatigue Index + stats match.
Alimente XGBoost prévention blessures et dashboard Load Management.';

COMMENT ON TABLE sport_metrics.mart.dim_players_info
IS 'Référentiel joueurs : biométrie, position, expérience.
60 joueurs actifs. Mise à jour hebdomadaire.';

COMMENT ON TABLE sport_metrics.mart.dim_calendrier_matchs
IS 'Référentiel matchs : dates, saisons, adversaires, domicile/extérieur.
389 matchs sur 5 saisons NBA (2019-2024).';

### Niveau 3: Tags Unity Catalog
Catégorisation par domaine, couche, audience et sensibilité RGPD.

In [0]:
-- ============================================
-- Tags tables RAW
-- ============================================
ALTER TABLE sport_metrics.raw.team_players_stats
SET TAGS ('layer' = 'raw', 'domain' = 'sport', 'refresh' = 'daily', 'status' = 'production');

ALTER TABLE sport_metrics.raw.team_training_sessions
SET TAGS ('layer' = 'raw', 'domain' = 'sport', 'refresh' = 'daily', 'status' = 'production');

ALTER TABLE sport_metrics.raw.team_players_personal_info
SET TAGS ('layer' = 'raw', 'domain' = 'sport', 'refresh' = 'weekly', 'status' = 'production');

ALTER TABLE sport_metrics.raw.team_games_dataset
SET TAGS ('layer' = 'raw', 'domain' = 'sport', 'refresh' = 'daily', 'status' = 'production');

ALTER TABLE sport_metrics.raw.team_boxscores
SET TAGS ('layer' = 'raw', 'domain' = 'sport', 'refresh' = 'daily', 'status' = 'production');

-- ============================================
-- Tags tables MART
-- ============================================
ALTER TABLE sport_metrics.mart.int_fatigue_index
SET TAGS ('layer' = 'intermediate', 'domain' = 'sport', 'refresh' = 'daily', 'status' = 'production');

ALTER TABLE sport_metrics.mart.mart_player
SET TAGS ('layer' = 'mart', 'domain' = 'sport', 'refresh' = 'daily', 'status' = 'production',
          'audience_coach' = 'true', 'owner' = 'data_team');

ALTER TABLE sport_metrics.mart.mart_game
SET TAGS ('layer' = 'mart', 'domain' = 'sport', 'refresh' = 'daily', 'status' = 'production',
          'audience_coach' = 'true', 'audience_direction' = 'true', 'owner' = 'data_team');

ALTER TABLE sport_metrics.mart.mart_physical_condition
SET TAGS ('layer' = 'mart', 'domain' = 'sante', 'refresh' = 'daily', 'status' = 'production',
          'audience_coach' = 'true', 'audience_staff_medical' = 'true',
          'pii' = 'true', 'owner' = 'data_team');

ALTER TABLE sport_metrics.mart.dim_players_info
SET TAGS ('layer' = 'dim', 'domain' = 'sport', 'refresh' = 'weekly', 'status' = 'production',
          'pii' = 'true', 'audience_coach' = 'true');

ALTER TABLE sport_metrics.mart.dim_calendrier_matchs
SET TAGS ('layer' = 'dim', 'domain' = 'sport', 'refresh' = 'daily', 'status' = 'production',
          'audience_coach' = 'true', 'audience_direction' = 'true');

In [0]:
-- ============================================
-- Tags colonnes PII - Conformité RGPD
-- ============================================
ALTER TABLE sport_metrics.mart.dim_players_info
ALTER COLUMN player_name SET TAGS ('pii' = 'true', 'sensitivity' = 'medium');

ALTER TABLE sport_metrics.mart.dim_players_info
ALTER COLUMN birthdate SET TAGS ('pii' = 'true', 'sensitivity' = 'high');

ALTER TABLE sport_metrics.mart.dim_players_info
ALTER COLUMN age SET TAGS ('pii' = 'true', 'sensitivity' = 'medium');

ALTER TABLE sport_metrics.mart.dim_players_info
ALTER COLUMN height_cm SET TAGS ('pii' = 'true', 'sensitivity' = 'low');

ALTER TABLE sport_metrics.mart.dim_players_info
ALTER COLUMN weight_kg SET TAGS ('pii' = 'true', 'sensitivity' = 'low');

-- ============================================
-- Tags colonnes ML Features
-- ============================================
ALTER TABLE sport_metrics.mart.mart_physical_condition
ALTER COLUMN fi_last_training SET TAGS ('ml_feature' = 'true', 'model' = 'xgboost');

ALTER TABLE sport_metrics.mart.mart_physical_condition
ALTER COLUMN fi_avg_7d SET TAGS ('ml_feature' = 'true', 'model' = 'xgboost');

ALTER TABLE sport_metrics.mart.mart_physical_condition
ALTER COLUMN fi_avg_28d SET TAGS ('ml_feature' = 'true', 'model' = 'xgboost');

ALTER TABLE sport_metrics.mart.mart_physical_condition
ALTER COLUMN training_load_7d SET TAGS ('ml_feature' = 'true', 'model' = 'xgboost');

ALTER TABLE sport_metrics.mart.mart_physical_condition
ALTER COLUMN injury_risk SET TAGS ('ml_feature' = 'true', 'model' = 'xgboost', 'target' = 'true');

### Niveau 4: Règles de Qualité
Contraintes CHECK sur les tables mart.
Garantissent l'intégrité des données avant consommation par Power BI.

In [0]:
-- ============================================
-- Nettoyage : suppression contraintes existantes
-- Pattern : DROP IF EXISTS → ADD
-- Garantit un état propre et reproductible
-- ============================================

-- mart_player
ALTER TABLE sport_metrics.mart.mart_player DROP CONSTRAINT IF EXISTS player_id_valid;
ALTER TABLE sport_metrics.mart.mart_player DROP CONSTRAINT IF EXISTS game_id_valid;
ALTER TABLE sport_metrics.mart.mart_player DROP CONSTRAINT IF EXISTS minutes_played_valid;
ALTER TABLE sport_metrics.mart.mart_player DROP CONSTRAINT IF EXISTS points_valid;
ALTER TABLE sport_metrics.mart.mart_player DROP CONSTRAINT IF EXISTS performance_score_valid;

-- mart_physical_condition
ALTER TABLE sport_metrics.mart.mart_physical_condition DROP CONSTRAINT IF EXISTS session_id_pc_valid;
ALTER TABLE sport_metrics.mart.mart_physical_condition DROP CONSTRAINT IF EXISTS fi_last_training_valid;
ALTER TABLE sport_metrics.mart.mart_physical_condition DROP CONSTRAINT IF EXISTS fi_avg_7d_valid;
ALTER TABLE sport_metrics.mart.mart_physical_condition DROP CONSTRAINT IF EXISTS fi_avg_28d_valid;
ALTER TABLE sport_metrics.mart.mart_physical_condition DROP CONSTRAINT IF EXISTS injury_risk_valid;
ALTER TABLE sport_metrics.mart.mart_physical_condition DROP CONSTRAINT IF EXISTS minutes_played_valid;
ALTER TABLE sport_metrics.mart.mart_physical_condition DROP CONSTRAINT IF EXISTS points_valid;
ALTER TABLE sport_metrics.mart.mart_physical_condition DROP CONSTRAINT IF EXISTS session_id_valid;

-- mart_game
ALTER TABLE sport_metrics.mart.mart_game DROP CONSTRAINT IF EXISTS game_id_game_valid;
ALTER TABLE sport_metrics.mart.mart_game DROP CONSTRAINT IF EXISTS game_id_valid;
ALTER TABLE sport_metrics.mart.mart_game DROP CONSTRAINT IF EXISTS win_pct_valid;
ALTER TABLE sport_metrics.mart.mart_game DROP CONSTRAINT IF EXISTS win_pct_range;
ALTER TABLE sport_metrics.mart.mart_game DROP CONSTRAINT IF EXISTS total_points_valid;
ALTER TABLE sport_metrics.mart.mart_game DROP CONSTRAINT IF EXISTS total_points_positive;

In [0]:
-- ============================================
-- Contraintes qualité - mart_player
-- ============================================
ALTER TABLE sport_metrics.mart.mart_player
ADD CONSTRAINT player_id_valid
CHECK (player_id IS NOT NULL AND player_id > 0);

ALTER TABLE sport_metrics.mart.mart_player
ADD CONSTRAINT game_id_valid
CHECK (game_id IS NOT NULL AND game_id > 0);

ALTER TABLE sport_metrics.mart.mart_player
ADD CONSTRAINT minutes_played_valid
CHECK (minutes_played IS NOT NULL AND minutes_played >= 0);

ALTER TABLE sport_metrics.mart.mart_player
ADD CONSTRAINT points_valid
CHECK (points IS NOT NULL AND points >= 0);

-- ============================================
-- Contraintes qualité - mart_physical_condition
-- ============================================
ALTER TABLE sport_metrics.mart.mart_physical_condition
ADD CONSTRAINT session_id_pc_valid
CHECK (session_id IS NOT NULL AND LENGTH(session_id) > 0);

ALTER TABLE sport_metrics.mart.mart_physical_condition
ADD CONSTRAINT fi_last_training_valid
CHECK (fi_last_training IS NOT NULL AND fi_last_training BETWEEN 0 AND 100);

ALTER TABLE sport_metrics.mart.mart_physical_condition
ADD CONSTRAINT fi_avg_7d_valid
CHECK (fi_avg_7d IS NOT NULL AND fi_avg_7d BETWEEN 0 AND 100);

ALTER TABLE sport_metrics.mart.mart_physical_condition
ADD CONSTRAINT injury_risk_valid
CHECK (injury_risk IS NOT NULL AND injury_risk BETWEEN 0 AND 1);

-- ============================================
-- Contraintes qualité - mart_game
-- ============================================
ALTER TABLE sport_metrics.mart.mart_game
ADD CONSTRAINT game_id_game_valid
CHECK (game_id IS NOT NULL AND game_id > 0);

ALTER TABLE sport_metrics.mart.mart_game
ADD CONSTRAINT win_pct_valid
CHECK (win_pct IS NOT NULL AND win_pct BETWEEN 0 AND 1);

ALTER TABLE sport_metrics.mart.mart_game
ADD CONSTRAINT total_points_valid
CHECK (total_points IS NOT NULL AND total_points >= 0);

In [0]:
-- Suppression des anciennes contraintes du Module 4
ALTER TABLE sport_metrics.mart.mart_player DROP CONSTRAINT IF EXISTS game_id_not_null;
ALTER TABLE sport_metrics.mart.mart_player DROP CONSTRAINT IF EXISTS minutes_played_positive;
ALTER TABLE sport_metrics.mart.mart_player DROP CONSTRAINT IF EXISTS player_id_not_null;
ALTER TABLE sport_metrics.mart.mart_player DROP CONSTRAINT IF EXISTS points_positive;
ALTER TABLE sport_metrics.mart.mart_player DROP CONSTRAINT IF EXISTS minutes_played_positi;

In [0]:
-- ============================================
-- Vérification de toutes les contraintes
-- ============================================
-- Voir les contraintes via la table système Delta
-- Vérification via DESCRIBE DETAIL
SHOW TBLPROPERTIES sport_metrics.mart.mart_physical_condition;